# Data Integration

This notebook integrates the cleaned electric vehicle registration data with the geographical lookup table and the mid-2024 population dataset. The integration is performed using Lower Layer Super Output Area (LSOA) codes to create a consistent England and Wales dataset for regional analysis.

The complete UK EV registration dataset is retained separately for national trend analysis and forecasting. The charging infrastructure dataset is loaded for later analysis but is not merged at this stage because it contains multiple geographical reporting levels.

In [1]:
from pathlib import Path

# Project folders
project_path = Path.cwd().parent
data_path = project_path / "Datasets"
processed_path = data_path / "processed"

print(processed_path)

/Users/adityakumar/Desktop/EV_Dissertation/Datasets/processed


In [2]:
# Import the required libraries

import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
# Set the project folders

project_folder = Path.cwd().parent

processed_folder = project_folder / "Datasets" / "processed"
merged_folder = project_folder / "Datasets" / "merged"

print("Processed folder:")
print(processed_folder)

print()

print("Merged folder:")
print(merged_folder)

Processed folder:
/Users/adityakumar/Desktop/EV_Dissertation/Datasets/processed

Merged folder:
/Users/adityakumar/Desktop/EV_Dissertation/Datasets/merged


In [4]:
# Load the cleaned datasets

ev_clean = pd.read_csv(processed_folder / "ev_clean.csv")

charging_clean = pd.read_csv(processed_folder / "charging_clean.csv")

lookup_clean = pd.read_csv(processed_folder / "lookup_clean.csv")

population_clean = pd.read_csv(processed_folder / "population_clean.csv")

print("All cleaned datasets loaded successfully.")

All cleaned datasets loaded successfully.


In [5]:
print("EV registrations:", ev_clean.shape)
print("Charging infrastructure:", charging_clean.shape)
print("Lookup table:", lookup_clean.shape)
print("Population:", population_clean.shape)

EV registrations: (285505, 62)
Charging infrastructure: (433, 29)
Lookup table: (35672, 4)
Population: (35672, 5)


## Inspect the Datasets Before Integration

Before integrating the datasets, the key variables and their structures are examined. This step ensures that the geographical identifiers required for merging are present and consistently named across all datasets.

In [6]:
# Display the first five rows of each dataset

print("EV Dataset")
display(ev_clean.head())

print("Lookup Dataset")
display(lookup_clean.head())

print("Population Dataset")
display(population_clean.head())

print("Charging Dataset")
display(charging_clean.head())

EV Dataset


,LSOA21CD,LSOA21NM,Fuel,Keepership,2026 Q1,2025 Q4,2025 Q3,2025 Q2,2025 Q1,2024 Q4,...,2014 Q1,2013 Q4,2013 Q3,2013 Q2,2013 Q1,2012 Q4,2012 Q3,2012 Q2,2012 Q1,2011 Q4
0,E01000001,City of London 001A,Battery electric,Company,13.0,10.0,10.0,10.0,9.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,E01000002,City of London 001B,Battery electric,Company,12.0,14.0,11.0,11.0,12.0,9.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
2,E01000005,City of London 001E,Battery electric,Company,18.0,18.0,22.0,21.0,18.0,18.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,E01000007,Barking and Dagenham 015A,Battery electric,Company,7.0,8.0,8.0,9.0,9.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,E01000008,Barking and Dagenham 015B,Battery electric,Company,134.0,129.0,125.0,124.0,117.0,99.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Lookup Dataset


,lsoa_code,lsoa_name,lad_code,lad_name
0,E01011949,Hartlepool 009A,E06000001,Hartlepool
1,E01011950,Hartlepool 008A,E06000001,Hartlepool
2,E01011951,Hartlepool 007A,E06000001,Hartlepool
3,E01011952,Hartlepool 002A,E06000001,Hartlepool
4,E01011953,Hartlepool 002B,E06000001,Hartlepool


Population Dataset


,lad_code,lad_name,lsoa_code,lsoa_name,population_2024
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1898
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1247
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1393
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1669
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2303


Charging Dataset


,area_code,area_name,Oct-19,Jan-20,Apr-20,Jul-20,Oct-20,Jan-21,Apr-21,Jul-21,...,Jan-24,Apr-24,Jul-24,Oct-24,Jan-25,Apr-25,Jul-25,Oct-25,Jan-26,Apr-26
0,K02000001,United Kingdom,15116.0,16505.0,17947.0,18265.0,19487.0,20775.0,22790.0,24374.0,...,53677.0,59670.0,64632.0,70042.0,73334.0,76507.0,82002.0,86021.0,87796.0,92141.0
1,K03000001,Great Britain,14821.0,16210.0,17642.0,17953.0,19169.0,20455.0,22463.0,24044.0,...,53212.0,59125.0,64014.0,69403.0,72654.0,75835.0,81318.0,85283.0,87069.0,91414.0
2,E92000001,England,12549.0,13719.0,14979.0,15395.0,16456.0,17459.0,19261.0,20563.0,...,46374.0,51506.0,55631.0,60364.0,63389.0,65877.0,70672.0,74115.0,75818.0,79198.0
3,E12000001,North East,738.0,752.0,786.0,812.0,849.0,820.0,854.0,887.0,...,1596.0,1942.0,1919.0,2295.0,2583.0,2553.0,2703.0,2698.0,2734.0,2950.0
4,E06000047,County Durham,92.0,96.0,102.0,105.0,106.0,110.0,121.0,116.0,...,291.0,309.0,313.0,359.0,400.0,462.0,482.0,482.0,489.0,523.0


In [7]:
# Display the column names for each dataset

print("EV Columns")
print(ev_clean.columns.tolist())

print("\nLookup Columns")
print(lookup_clean.columns.tolist())

print("\nPopulation Columns")
print(population_clean.columns.tolist())

print("\nCharging Columns")
print(charging_clean.columns.tolist())

EV Columns
['LSOA21CD', 'LSOA21NM', 'Fuel', 'Keepership', '2026 Q1', '2025 Q4', '2025 Q3', '2025 Q2', '2025 Q1', '2024 Q4', '2024 Q3', '2024 Q2', '2024 Q1', '2023 Q4', '2023 Q3', '2023 Q2', '2023 Q1', '2022 Q4', '2022 Q3', '2022 Q2', '2022 Q1', '2021 Q4', '2021 Q3', '2021 Q2', '2021 Q1', '2020 Q4', '2020 Q3', '2020 Q2', '2020 Q1', '2019 Q4', '2019 Q3', '2019 Q2', '2019 Q1', '2018 Q4', '2018 Q3', '2018 Q2', '2018 Q1', '2017 Q4', '2017 Q3', '2017 Q2', '2017 Q1', '2016 Q4', '2016 Q3', '2016 Q2', '2016 Q1', '2015 Q4', '2015 Q3', '2015 Q2', '2015 Q1', '2014 Q4', '2014 Q3', '2014 Q2', '2014 Q1', '2013 Q4', '2013 Q3', '2013 Q2', '2013 Q1', '2012 Q4', '2012 Q3', '2012 Q2', '2012 Q1', '2011 Q4']

Lookup Columns
['lsoa_code', 'lsoa_name', 'lad_code', 'lad_name']

Population Columns
['lad_code', 'lad_name', 'lsoa_code', 'lsoa_name', 'population_2024']

Charging Columns
['area_code', 'area_name', 'Oct-19', 'Jan-20', 'Apr-20', 'Jul-20', 'Oct-20', 'Jan-21', 'Apr-21', 'Jul-21', 'Oct-21', 'Jan-22', 'A

## Standardise the Geographical Identifiers

The geographical identifier columns are standardised before merging the datasets. Using consistent variable names reduces the risk of matching errors and simplifies the integration process.

In [8]:
# Rename the EV geographical columns

ev_clean = ev_clean.rename(
    columns={
        "LSOA21CD": "lsoa_code",
        "LSOA21NM": "lsoa_name"
    }
)

print("EV geographical columns renamed successfully.")

EV geographical columns renamed successfully.


In [9]:
# Check the updated column names

print(ev_clean.columns[:6].tolist())

['lsoa_code', 'lsoa_name', 'Fuel', 'Keepership', '2026 Q1', '2025 Q4']


## Prepare EV Data for Regional Integration

The EV registration dataset covers the whole United Kingdom, whereas the available LSOA lookup and population datasets cover England and Wales. Therefore, the complete EV dataset is retained for national trend analysis, while records with English and Welsh geographical codes are selected for detailed regional integration.

Scottish, Northern Irish and non-geographical records are excluded only from the Local Authority District analysis because equivalent geographical lookup and population variables are not available within the supporting datasets used in this study.

In [10]:
# Retain the complete EV dataset for UK-level analysis

ev_uk = ev_clean.copy()

print("UK EV records:", len(ev_uk))

UK EV records: 285505


In [11]:
# Select England and Wales records for regional integration

ev_ew = ev_clean[
    ev_clean["lsoa_code"]
    .astype(str)
    .str.startswith(("E", "W"))
].copy()

print("England and Wales EV records:", len(ev_ew))
print("Excluded regional records:", len(ev_clean) - len(ev_ew))

England and Wales EV records: 247667
Excluded regional records: 37838


## Merge EV Registrations with the Geography Lookup

The England and Wales EV records are merged with the geographical lookup using the LSOA code. The LSOA code is used as the authoritative geographical key, while the names from the lookup table provide standardised geographical labels.

In [12]:
# Merge England and Wales EV records with the lookup table

ev_lookup = ev_ew.merge(
    lookup_clean,
    on="lsoa_code",
    how="left",
    suffixes=("_ev", "_lookup"),
    validate="many_to_one"
)

print("Merge completed successfully.")

Merge completed successfully.


In [13]:
# Validate the geographical merge

print("Rows before merge:", len(ev_ew))
print("Rows after merge:", len(ev_lookup))
print("Missing LAD codes:", ev_lookup["lad_code"].isna().sum())

Rows before merge: 247667
Rows after merge: 247667
Missing LAD codes: 0


## Merge the Population Dataset

The population dataset is merged with the EV and geography data using the LSOA code. This adds the mid-2024 population estimate required to calculate population-standardised EV adoption indicators.

In [14]:
# Select only the population variables needed for the merge

population_merge = population_clean[
    [
        "lsoa_code",
        "population_2024"
    ]
].copy()

print("Population records:", len(population_merge))
print(
    "Duplicate LSOA codes:",
    population_merge["lsoa_code"].duplicated().sum()
)

Population records: 35672
Duplicate LSOA codes: 0


In [15]:
# Merge the population data using the LSOA code

ev_population = ev_lookup.merge(
    population_merge,
    on="lsoa_code",
    how="left",
    validate="many_to_one"
)

print("Population merge completed successfully.")

Population merge completed successfully.


In [16]:
# Validate the population merge

print("Rows before merge:", len(ev_lookup))
print("Rows after merge:", len(ev_population))
print(
    "Missing population values:",
    ev_population["population_2024"].isna().sum()
)

Rows before merge: 247667
Rows after merge: 247667
Missing population values: 0


# Summary

The cleaned EV registration data were successfully integrated with the geographical lookup table and mid-2024 population estimates using LSOA codes. The resulting integrated dataset provides a consistent England and Wales database for Local Authority District analysis.

The complete UK EV registration dataset has been retained separately for national trend analysis and forecasting, while the charging infrastructure dataset will be analysed independently in the subsequent notebooks.

In [17]:
# Save the integrated dataset

output_path = processed_path / "ev_integrated.csv"

ev_population.to_csv(output_path, index=False)

print("Integrated dataset saved successfully.")
print(output_path)

Integrated dataset saved successfully.
/Users/adityakumar/Desktop/EV_Dissertation/Datasets/processed/ev_integrated.csv
